# Encoding

## Objective

In this notebook, the feature-engineered dataset is prepared for machine learning by converting categorical variables into numerical representations.

Workflow:

- Load raw dataset
- Apply data cleaning
- Apply feature engineering
- Identify categorical variables
- Apply encoding
- Save encoded dataset

In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [5]:
from src.data_loader import load_data
from src.preprocessing import clean_data
from src.feature_engineering import engineer_features

In [6]:
df_raw = load_data("laptop_price - dataset.csv")

df_clean = clean_data(df_raw)

df_featured = engineer_features(df_clean)

df_featured.head()

,company,product,type_name,inches,cpu_company,cpu_frequency,ram,gpu_company,operating_system,weight,...,hdd,flash_storage,hybrid,resolution_width,resolution_height,ips_panel,touchscreen,cpu_family,gpu_family,ppi
0,Apple,MacBook Pro,Ultrabook,13.3,Intel,2.3,8,Intel,macOS,1.37,...,0.0,0.0,0.0,2560,1600,1,0,Core i5,Iris Graphics,226.983005
1,Apple,Macbook Air,Ultrabook,13.3,Intel,1.8,8,Intel,macOS,1.34,...,0.0,128.0,0.0,1440,900,0,0,Core i5,HD Graphics,127.677940
2,HP,250 G6,Notebook,15.6,Intel,2.5,8,Intel,No OS,1.86,...,0.0,0.0,0.0,1920,1080,0,0,Core i5,HD Graphics,141.211998
3,Apple,MacBook Pro,Ultrabook,15.4,Intel,2.7,16,AMD,macOS,1.83,...,0.0,0.0,0.0,2880,1800,1,0,Core i7,Radeon,220.534624
4,Apple,MacBook Pro,Ultrabook,13.3,Intel,3.1,8,Intel,macOS,1.37,...,0.0,0.0,0.0,2560,1600,1,0,Core i5,Iris Graphics,226.983005


In [7]:
df_featured.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   company            1275 non-null   str    
 1   product            1275 non-null   str    
 2   type_name          1275 non-null   str    
 3   inches             1275 non-null   float64
 4   cpu_company        1275 non-null   str    
 5   cpu_frequency      1275 non-null   float64
 6   ram                1275 non-null   int64  
 7   gpu_company        1275 non-null   str    
 8   operating_system   1275 non-null   str    
 9   weight             1275 non-null   float64
 10  price              1275 non-null   float64
 11  ssd                1275 non-null   float64
 12  hdd                1275 non-null   float64
 13  flash_storage      1275 non-null   float64
 14  hybrid             1275 non-null   float64
 15  resolution_width   1275 non-null   int64  
 16  resolution_height  1275 non-null   

In [8]:
df_featured.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
company,1275,19,Dell,291,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product,1275,618,XPS 13,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_name,1275,6,Notebook,707,NaN,NaN,NaN,NaN,NaN,NaN,NaN
inches,1275.0,NaN,NaN,NaN,15.022902,1.42947,10.1,14.0,15.6,15.6,18.4
cpu_company,1275,3,Intel,1214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpu_frequency,1275.0,NaN,NaN,NaN,2.30298,0.503846,0.9,2.0,2.5,2.7,3.6
ram,1275.0,NaN,NaN,NaN,8.440784,5.097809,2.0,4.0,8.0,8.0,64.0
gpu_company,1275,4,Intel,704,NaN,NaN,NaN,NaN,NaN,NaN,NaN
operating_system,1275,9,Windows 10,1048,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,1275.0,NaN,NaN,NaN,2.040525,0.669196,0.69,1.5,2.04,2.31,4.7


In [10]:
# Identifying Features
categorical_columns = df_featured.select_dtypes(include="object").columns.tolist()

numerical_columns = df_featured.select_dtypes(exclude="object").columns.tolist()

print(f"Categorical Features ({len(categorical_columns)}):")
print(categorical_columns)

print()

print(f"Numerical Features ({len(numerical_columns)}):")
print(numerical_columns)

Categorical Features (8):
['company', 'product', 'type_name', 'cpu_company', 'gpu_company', 'operating_system', 'cpu_family', 'gpu_family']

Numerical Features (14):
['inches', 'cpu_frequency', 'ram', 'weight', 'price', 'ssd', 'hdd', 'flash_storage', 'hybrid', 'resolution_width', 'resolution_height', 'ips_panel', 'touchscreen', 'ppi']


In [11]:
df_featured["product"].nunique()

618

In [12]:
df_featured["product"].value_counts().head(20)

product
XPS 13                30
Inspiron 3567         25
250 G6                21
Legion Y520-15IKBN    19
Vostro 3568           19
Inspiron 5570         18
ProBook 450           18
Alienware 17          15
Inspiron 5567         14
Satellite Pro         13
Aspire 3              12
EliteBook 840         12
ThinkPad X1           12
Latitude 5580         12
MacBook Pro           10
Inspiron 7567         10
EliteBook 850         10
XPS 15                 9
EliteBook 820          9
ProBook 470            8
Name: count, dtype: int64

In [13]:
df_featured["company"].nunique()

19

In [14]:
df_featured["cpu_company"].nunique()

3

In [15]:
df_featured["gpu_company"].nunique()

4

In [16]:
df_featured["type_name"].nunique()

6

In [17]:
df_featured["operating_system"].nunique()

9

In [19]:
df_featured["cpu_family"].nunique()

8

In [20]:
df_featured["gpu_family"].nunique()

9

# Encoding Strategy

Before applying any encoding technique, we analyze each categorical feature and select the most appropriate encoding method.

The goal is to preserve useful information while avoiding unnecessary dimensionality.

In [21]:
encoding_strategy = pd.DataFrame(
    {
        "Feature": categorical_columns,
        "Unique Values": [
            df_featured[col].nunique()
            for col in categorical_columns
        ],
    }
)

encoding_strategy

,Feature,Unique Values
0,company,19
1,product,618
2,type_name,6
3,cpu_company,3
4,gpu_company,4
5,operating_system,9
6,cpu_family,8
7,gpu_family,9


In [22]:
encoding_strategy["Encoding"] = [
    "One-Hot",
    "Drop",
    "One-Hot",
    "One-Hot",
    "One-Hot",
    "One-Hot",
    "One-Hot",
    "One-Hot",
]

encoding_strategy

,Feature,Unique Values,Encoding
0,company,19,One-Hot
1,product,618,Drop
2,type_name,6,One-Hot
3,cpu_company,3,One-Hot
4,gpu_company,4,One-Hot
5,operating_system,9,One-Hot
6,cpu_family,8,One-Hot
7,gpu_family,9,One-Hot


In [23]:
df_encoded = df_featured.drop(columns="product")

df_encoded.head()

,company,type_name,inches,cpu_company,cpu_frequency,ram,gpu_company,operating_system,weight,price,...,hdd,flash_storage,hybrid,resolution_width,resolution_height,ips_panel,touchscreen,cpu_family,gpu_family,ppi
0,Apple,Ultrabook,13.3,Intel,2.3,8,Intel,macOS,1.37,1339.69,...,0.0,0.0,0.0,2560,1600,1,0,Core i5,Iris Graphics,226.983005
1,Apple,Ultrabook,13.3,Intel,1.8,8,Intel,macOS,1.34,898.94,...,0.0,128.0,0.0,1440,900,0,0,Core i5,HD Graphics,127.677940
2,HP,Notebook,15.6,Intel,2.5,8,Intel,No OS,1.86,575.00,...,0.0,0.0,0.0,1920,1080,0,0,Core i5,HD Graphics,141.211998
3,Apple,Ultrabook,15.4,Intel,2.7,16,AMD,macOS,1.83,2537.45,...,0.0,0.0,0.0,2880,1800,1,0,Core i7,Radeon,220.534624
4,Apple,Ultrabook,13.3,Intel,3.1,8,Intel,macOS,1.37,1803.60,...,0.0,0.0,0.0,2560,1600,1,0,Core i5,Iris Graphics,226.983005


In [24]:
df_encoded.columns

Index(['company', 'type_name', 'inches', 'cpu_company', 'cpu_frequency', 'ram',
       'gpu_company', 'operating_system', 'weight', 'price', 'ssd', 'hdd',
       'flash_storage', 'hybrid', 'resolution_width', 'resolution_height',
       'ips_panel', 'touchscreen', 'cpu_family', 'gpu_family', 'ppi'],
      dtype='str')

In [25]:
categorical_columns = [
    "company",
    "type_name",
    "cpu_company",
    "gpu_company",
    "operating_system",
    "cpu_family",
    "gpu_family",
]

df_encoded = pd.get_dummies(
    df_encoded,
    columns=categorical_columns,
    dtype=int
)

df_encoded.head()

,inches,cpu_frequency,ram,weight,price,ssd,hdd,flash_storage,hybrid,resolution_width,...,cpu_family_Pentium,gpu_family_FirePro,gpu_family_GTX,gpu_family_HD Graphics,gpu_family_Iris Graphics,gpu_family_MX,gpu_family_Other,gpu_family_Quadro,gpu_family_Radeon,gpu_family_UHD Graphics
0,13.3,2.3,8,1.37,1339.69,128.0,0.0,0.0,0.0,2560,...,0,0,0,0,1,0,0,0,0,0
1,13.3,1.8,8,1.34,898.94,0.0,0.0,128.0,0.0,1440,...,0,0,0,1,0,0,0,0,0,0
2,15.6,2.5,8,1.86,575.00,256.0,0.0,0.0,0.0,1920,...,0,0,0,1,0,0,0,0,0,0
3,15.4,2.7,16,1.83,2537.45,512.0,0.0,0.0,0.0,2880,...,0,0,0,0,0,0,0,0,1,0
4,13.3,3.1,8,1.37,1803.60,256.0,0.0,0.0,0.0,2560,...,0,0,0,0,1,0,0,0,0,0


In [26]:
df_encoded.shape

(1275, 72)

In [27]:
df_encoded.columns.tolist()

['inches',
 'cpu_frequency',
 'ram',
 'weight',
 'price',
 'ssd',
 'hdd',
 'flash_storage',
 'hybrid',
 'resolution_width',
 'resolution_height',
 'ips_panel',
 'touchscreen',
 'ppi',
 'company_Acer',
 'company_Apple',
 'company_Asus',
 'company_Chuwi',
 'company_Dell',
 'company_Fujitsu',
 'company_Google',
 'company_HP',
 'company_Huawei',
 'company_LG',
 'company_Lenovo',
 'company_MSI',
 'company_Mediacom',
 'company_Microsoft',
 'company_Razer',
 'company_Samsung',
 'company_Toshiba',
 'company_Vero',
 'company_Xiaomi',
 'type_name_2 in 1 Convertible',
 'type_name_Gaming',
 'type_name_Netbook',
 'type_name_Notebook',
 'type_name_Ultrabook',
 'type_name_Workstation',
 'cpu_company_AMD',
 'cpu_company_Intel',
 'cpu_company_Samsung',
 'gpu_company_AMD',
 'gpu_company_ARM',
 'gpu_company_Intel',
 'gpu_company_Nvidia',
 'operating_system_Android',
 'operating_system_Chrome OS',
 'operating_system_Linux',
 'operating_system_Mac OS X',
 'operating_system_No OS',
 'operating_system_Window

In [28]:
len(df_encoded.columns)

72

In [29]:
from src.encoding import encode_features

In [30]:
df_raw = load_data("laptop_price - dataset.csv")

df_clean = clean_data(df_raw)

df_featured = engineer_features(df_clean)

df_encoded = encode_features(df_featured)

df_encoded.head()

,inches,cpu_frequency,ram,weight,price,ssd,hdd,flash_storage,hybrid,resolution_width,...,cpu_family_Pentium,gpu_family_FirePro,gpu_family_GTX,gpu_family_HD Graphics,gpu_family_Iris Graphics,gpu_family_MX,gpu_family_Other,gpu_family_Quadro,gpu_family_Radeon,gpu_family_UHD Graphics
0,13.3,2.3,8,1.37,1339.69,128.0,0.0,0.0,0.0,2560,...,0,0,0,0,1,0,0,0,0,0
1,13.3,1.8,8,1.34,898.94,0.0,0.0,128.0,0.0,1440,...,0,0,0,1,0,0,0,0,0,0
2,15.6,2.5,8,1.86,575.00,256.0,0.0,0.0,0.0,1920,...,0,0,0,1,0,0,0,0,0,0
3,15.4,2.7,16,1.83,2537.45,512.0,0.0,0.0,0.0,2880,...,0,0,0,0,0,0,0,0,1,0
4,13.3,3.1,8,1.37,1803.60,256.0,0.0,0.0,0.0,2560,...,0,0,0,0,1,0,0,0,0,0


In [31]:
df_encoded.shape

(1275, 72)

In [32]:
df_encoded.equals(
    encode_features(engineer_features(clean_data(load_data("laptop_price - dataset.csv"))))
)

True

In [33]:
PROCESSED_DATA_PATH = Path("../data/processed/laptop_encoded.csv")

df_encoded.to_csv(PROCESSED_DATA_PATH, index=False)

print("Encoded dataset saved successfully!")

Encoded dataset saved successfully!


# Summary

In this notebook:

- Loaded the raw dataset
- Applied data cleaning
- Applied feature engineering
- Selected an encoding strategy
- Removed the high-cardinality `product` feature
- Applied One-Hot Encoding to categorical variables
- Saved the encoded dataset for machine learning